###  **Title: DSC670-T301 Advanced Uses of Generative AI (2271-1)**
####  **Professor:** *Frank Neugebauer*
####  **Date: 29 Sep 2026**
####  **Author:** *Akhil Anand Thazhaniyil Muralidharan*
####  **Description:** Week4 Exercise : Prompt-Based Solutions — Zero-Shot, One-Shot, and Chain of Thought.

*************

### Description
This notebook demonstrates three prompt-engineering approaches using the OpenAI API:

1. Zero-shot prompting for extracting order information from an email
2. One-shot prompting for applying the same extraction format to a second email
3. Chat completion with worked examples for solving a mathematical reasoning problem

### Approach

**Zero-shot**: I just tell the model what to do — no examples. So just giving the input and ask the model to do certain steps, it has to figure out the structure on its own.

**One-shot**: I give the model one complete example first, then ask it to do the same thing with new data. In the exercise I am giving output of first case as input to the one shot prompt. 

**Chain of thought**: I show the model how to reason step-by-step, then ask it a new question expecting the same reasoning pattern.

*************

### Import Required Libraries

In [1]:
import json
import re
from openai import OpenAI

### Setup the Client

In [ ]:
# OpenAI API Configuration
OPENAI_API_KEY = ""
#Declare model
MODEL_NAME = "gpt-4o"

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client is ready.")

OpenAI client is ready.


### Problem #1: Math and Extraction with Zero and One-Shot Prompting

* **Input** : Email Sample
* **Prompt** : Asking the model to extract fields, not giving an example of what the output should look like.
* **Output** : Price is not available so model will have to make up reasonable shoe prices based on its training data. Perform artthmetic operations to derive subtotals, grand total. etc. Final output should be in JSON format.

In [3]:
#Declaring Emails
email1="""Hey Jim,



I'd like to order some new shoes. Please ship the following:



1. Nike Air Jordan I - 2 pair

2. Converse All-Star - 10 pair

3. New Balance 990 - 1 pair

4. Nike Zoom Fly 5 men's red - 2 pair



Please provide sub-totals and grand total cost.



Thanks.



Lance Gentry

123 Main St.

Chelsea, MI 48109

248-229-2229

"""
email2="""Hey Michelle,



I'd like to order some new shoes. Please ship the following:



1. Nike Air Jordan I - 1 pair

2. Converse All-Star - 20 pair

3. New Balance 990 - 2 pair

4. Nike Zoom Fly 5 men's red - 5 pair



Please provide sub-totals and grand total cost.



Thanks.



Artis Gilmore

723 Lexington Blvd.

New York, NY 10001

(503) 484-1029"""

### Part A: Zero-Shot Prompt

A zero-shot prompt describes the task without showing a completed example. I explicitly list the required fields and calculation rules, but I allow the model to decide the prices. This approach tests whether the instructions alone provide enough guidance for a valid JSON response.

The main limitation is price variability. Since the email does not contain a price list, the model must generate illustrative values. The important evaluation points are whether the selected prices are applied consistently, whether the subtotals are correct, and whether the JSON includes every requested field.

In [4]:
prompt_zs=f"""
Extract all requested order information from the email and return only one valid JSON object.
Do not include Markdown fences, introductory text, or explanations.

Required fields:
- shoe_agent_name: the person addressed in the greeting
- request_type: use "Order"
- shoes: an array containing shoe_brand, shoe_model, shoe_quantity,
  shoe_price, and shoe_subtotal for each item
- grand_total
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone

Instructions:
1. Assign a reasonable illustrative retail price to each shoe model.
2. Use the same price for a model wherever that model appears in this order.
3. Calculate shoe_subtotal as shoe_quantity multiplied by shoe_price.
4. Calculate grand_total as the sum of all shoe_subtotal values.
5. Use numbers, not currency strings, for price and total fields.
6. Preserve the customer phone number exactly as written.

Email:
{email1}
"""

print(f"Zero Shot Prompt:\n{prompt_zs}")

Zero Shot Prompt:

Extract all requested order information from the email and return only one valid JSON object.
Do not include Markdown fences, introductory text, or explanations.

Required fields:
- shoe_agent_name: the person addressed in the greeting
- request_type: use "Order"
- shoes: an array containing shoe_brand, shoe_model, shoe_quantity,
  shoe_price, and shoe_subtotal for each item
- grand_total
- customer_name
- customer_street
- customer_city
- customer_state
- customer_zip
- customer_phone

Instructions:
1. Assign a reasonable illustrative retail price to each shoe model.
2. Use the same price for a model wherever that model appears in this order.
3. Calculate shoe_subtotal as shoe_quantity multiplied by shoe_price.
4. Calculate grand_total as the sum of all shoe_subtotal values.
5. Use numbers, not currency strings, for price and total fields.
6. Preserve the customer phone number exactly as written.

Email:
Hey Jim,



I'd like to order some new shoes. Please ship the 

In [5]:
response_zs = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are a precise information-extraction assistant."
        },
        {
            "role": "user",
            "content": prompt_zs
        }
    ],
    temperature=0
)

zero_shot_result = response_zs.choices[0].message.content
cleaned_result = zero_shot_result.strip()
cleaned_result = re.sub(r"^```(?:json)?\s*", "", cleaned_result, flags=re.IGNORECASE)
cleaned_result = re.sub(r"\s*```$", "", cleaned_result)
zero_shot_json = json.loads(cleaned_result)

print("COMPLETE ZERO-SHOT JSON STRUCTURE")
print(json.dumps(zero_shot_json, indent=2))

output_filename = "zero_shot_result.json"
with open(output_filename, "w", encoding="utf-8") as output_file:
    json.dump(zero_shot_json, output_file, indent=2)

print(f"JSON file generated: {output_filename}")


COMPLETE ZERO-SHOT JSON STRUCTURE
{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 150,
      "shoe_subtotal": 300
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 50,
      "shoe_subtotal": 500
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 180,
      "shoe_subtotal": 180
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160,
      "shoe_subtotal": 320
    }
  ],
  "grand_total": 1300,
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229"
}
JSON file generated: zero_shot_result.json


In [6]:
def validate_order_math(order_json):
    """Check item subtotals and the grand total independently in Python."""
    issues = []
    calculated_total = 0

    for position, shoe in enumerate(order_json["shoes"], start=1):
        expected_subtotal = shoe["shoe_quantity"] * shoe["shoe_price"]
        calculated_total += expected_subtotal

        if shoe["shoe_subtotal"] != expected_subtotal:
            issues.append(
                f"Item {position}: expected subtotal {expected_subtotal}, "
                f"but received {shoe['shoe_subtotal']}."
            )

    if order_json["grand_total"] != calculated_total:
        issues.append(
            f"Expected grand total {calculated_total}, "
            f"but received {order_json['grand_total']}."
        )

    return {
        "valid": len(issues) == 0,
        "calculated_grand_total": calculated_total,
        "issues": issues
    }

print("ARITHMETIC VALIDATION")
print(json.dumps(validate_order_math(zero_shot_json), indent=2))


ARITHMETIC VALIDATION
{
  "valid": true,
  "calculated_grand_total": 1300,
  "issues": []
}


### Comments

* Response correctly created based on the prompt 
    * Identified Jim as the shoe agent
    * classify the request as an order
    * extract Lance Gentry's contact information
    * four shoe objects with correct quantities. 
    * Created price, subtotal and grant total as requested.

I did the separate validation step as while creating JSON does not guarantee correct arithmetic. Did the validation result to confirm whether every subtotal equals quantity multiplied by price and whether the grand total equals the sum of the subtotals.
**************

### Part B: One-Shot Prompt

Unlike zero-shot prompting, where only instructions are provided, one-shot prompting includes a complete example of the task. By showing the model how the first email was transformed into the desired output, it can apply the same pattern when processing the second email.

In [7]:
# One-shot prompt
prompt_os = f"""I will show you an example of extracting order information from an email as JSON. 
Then I'll give you a new email to extract in the same format.

EXAMPLE
Email:
{email1}

Extracted JSON:
{json.dumps(zero_shot_json, indent=2)}

NOW EXTRACT THIS EMAIL IN THE SAME FORMAT
Email:
{email2}

Requirements:
1. Return only one valid JSON object with no Markdown or explanation.
2. Use exactly the same JSON structure and field names as the example.
3. Use the same price for each shoe model as shown in the example.
4. Recalculate every subtotal using the target quantities.
5. Recalculate the grand total.
6. Extract customer and agent information only from the target email.
7. Preserve the target phone number exactly as written."""

print(f"One-shot prompt:\n{prompt_os}")

One-shot prompt:
I will show you an example of extracting order information from an email as JSON. 
Then I'll give you a new email to extract in the same format.

EXAMPLE
Email:
Hey Jim,



I'd like to order some new shoes. Please ship the following:



1. Nike Air Jordan I - 2 pair

2. Converse All-Star - 10 pair

3. New Balance 990 - 1 pair

4. Nike Zoom Fly 5 men's red - 2 pair



Please provide sub-totals and grand total cost.



Thanks.



Lance Gentry

123 Main St.

Chelsea, MI 48109

248-229-2229



Extracted JSON:
{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 150,
      "shoe_subtotal": 300
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 50,
      "shoe_subtotal": 500
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1

In [8]:
response_os = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are a precise information-extraction assistant."
        },
        {
            "role": "user",
            "content": prompt_os
        }
    ],
    temperature=0
)

one_shot_result = response_os.choices[0].message.content
cleaned_result_os = one_shot_result.strip()
cleaned_result_os = re.sub(r"^```(?:json)?\s*", "", cleaned_result_os, flags=re.IGNORECASE)
cleaned_result_os = re.sub(r"\s*```$", "", cleaned_result_os)
one_shot_json = json.loads(cleaned_result_os)

print("COMPLETE ONE-SHOT JSON STRUCTURE")
print(json.dumps(one_shot_json, indent=2))

output_filename_os = "one_shot_result.json"
with open(output_filename_os, "w", encoding="utf-8") as output_file_os:
    json.dump(one_shot_json, output_file_os, indent=2)

print(f"JSON file generated: {output_filename_os}")


COMPLETE ONE-SHOT JSON STRUCTURE
{
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 150,
      "shoe_subtotal": 150
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 50,
      "shoe_subtotal": 1000
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 2,
      "shoe_price": 180,
      "shoe_subtotal": 360
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160,
      "shoe_subtotal": 800
    }
  ],
  "grand_total": 2310,
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029"
}
JSON file generated: one_shot_resul

In [9]:
print(json.dumps(validate_order_math(one_shot_json), indent=2))

{
  "valid": true,
  "calculated_grand_total": 2310,
  "issues": []
}


### Comparing Zero-Shot and One-Shot Results
### Similarities
* Both prompting extracts the same requested information: agent name, request type, four shoe items, customer details, item prices, subtotals, and the grand total.
* Both responses generated in valid JSON format.
* Both methods preserve the customer contact information exactly as it appears in the source email.
* Both outputs are evaluated using the same Python validation function to verify structure and calculations.
* Both approaches aim to transform unstructured email content into a structured and machine-readable format.
* Both response generated correctly-
    * The first email is addressed to Jim and contains Lance Gentry's order with quantities 2, 10, 1, and 2.
    * The second email is addressed to Michelle and contains Artis Gilmore's order with quantities 1, 20, 2, and 5.
    * The second email presents the phone number using parentheses, providing an additional test for exact extraction and formatting preservation.
### Differences Identified
* The zero-shot prompt provides only instructions, while the one-shot prompt includes a complete input-output example for guidance.
* In the zero-shot approach, the model independently determines the expected structure and illustrative pricing based on the instructions provided.
* One-shot prompting generally offers greater control over output consistency, whereas zero-shot prompting is more flexible and uses fewer prompt tokens.

I performed all validation checks, including JSON key comparison, field ordering and nesting verification, product price validation, phone number format verification, subtotal and grand total arithmetic validation, and customer data extraction accuracy. The results indicate that both the zero-shot and one-shot outputs are correct and successfully pass all validation criteria. The one-shot approach provided slightly greater consistency and predictability in formatting, while the zero-shot approach achieved comparable accuracy with a lower prompt overhead.
*************


### Problem #2: Chat Completion for Chain of Thought
* Chain of thought (CoT) prompting is a technique where you show the model how to reason step-by-step through example problems. Then when you ask a new question, the model follows the same reasoning pattern.

* In this exercise provides three solved math problems as examples, then asks: "When I was 6 my sister was half my age. Now I'm 70 how old is my sister?"
    * This is a classic trick question. Many people answer 35 — but the correct answer is 67. Let's see if chain-of-thought helps the model get it right.


In [10]:
import html
from IPython.display import HTML, display

# Chain of Thought - using chat messages format
# The previous Q&A pairs serve as examples of step-by-step reasoning

messages = [
    {"role": "system", "content": "You are a helpful assistant that solves math problems step by step, showing your reasoning clearly."},
    
    # Example 1
    {"role": "user", "content": "There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?"},
    {"role": "assistant", "content": "There are 4 days from Monday to Thursday. 5 computers were added each day. That means in total 4 * 5 = 20 computers were added. There were 9 computers initially, so now there are 9 + 20 = 29 computers. The answer is 29."},
    
    # Example 2
    {"role": "user", "content": "Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?"},
    {"role": "assistant", "content": "Michael initially had 58 balls. He lost 23 on Tuesday, so after that he has 58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answer is 33."},
    
    # Example 3
    {"role": "user", "content": "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"},
    {"role": "assistant", "content": "She bought 5 bagels for $3 each. This means she spent $15. She has $8 left."},
    
    # The actual question
    {"role": "user", "content": "When I was 6 my sister was half my age. Now I'm 70 how old is my sister?"}
]

cot_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    temperature=0
)

cot_result = cot_response.choices[0].message.content
wrapped_result = html.escape(cot_result)
display(HTML(
    '<div style="white-space: pre-wrap; overflow-wrap: anywhere; '
    'word-break: normal; max-width: 100%;">'
    f'<strong>WORKED-EXAMPLE CHAT RESULT</strong><br><br>{wrapped_result}'
    '</div>'
))


## Direct Prompt Without the Worked Examples

I also run the final question without the earlier question-and-answer pairs. This creates a useful comparison. Both calls may produce the correct answer, especially with a capable model, but the worked-example conversation should more strongly encourage the requested explanatory pattern.

In [11]:
direct_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "Answer the math question accurately and concisely."
        },
        {
            "role": "user",
            "content": (
                "When I was 6 my sister was half my age. "
                "Now I'm 70. How old is my sister?"
            )
        }
    ],
    temperature=0
)

direct_result = direct_response.choices[0].message.content
wrapped_result_dir = html.escape(direct_result)
display(HTML(
    '<div style="white-space: pre-wrap; overflow-wrap: anywhere; '
    'word-break: normal; max-width: 100%;">'
    f'<strong>WORKED-EXAMPLE CHAT RESULT</strong><br><br>{wrapped_result_dir}'
    '</div>'
))

## Comments

* Computed the correct answer, 67. The sister was 3 when the speaker was 6, making her 3 years younger. The three-year difference remains constant, so `70 - 3 = 67`.

* The worked-example response is expected to follow the earlier answer pattern by stating the sister's original age, calculating the age difference, applying it to the current age, and ending with a direct answer. The direct prompt may reach the same result, but it has less guidance about response structure. Did validations for correctness, clarity and amount of explanation.
****************

# Conclusion

This exercise demonstrates that prompt design influences structure, consistency, and explanation. Zero-shot prompting is useful when the instructions are clear and no demonstration is available, but the output may vary when information such as product prices is missing. One-shot prompting reduces this ambiguity by giving the model a complete pattern to follow. It is especially useful when another program will consume the JSON and expects stable field names and nesting.

The math exercise shows how previous chat turns can guide the style used for a new problem. The worked examples do not replace the need to interpret the final question correctly, but they encourage the model to make its calculation visible and checkable. The most important lesson is that model output should still be validated. Clear prompts improve results, while programmatic checks provide an additional layer of reliability.
************

## References

OpenAI. (n.d.). *OpenAI API documentation*. https://platform.openai.com/docs/

OpenAI. (n.d.). *OpenAI Python library*. https://github.com/openai/openai-python